In [ ]:
# ---------------------------------------------------------
# CELL PURPOSE: Set up the local Ollama model service and environment
# This cell installs required Python packages, downloads or installs the Ollama binary,
,
,
,
,
,
,
,
,
,
,
,
🧹 Flushing stale network ports and ghost daemons...")
!fuser -k 11434/tcp 2>/dev/null
!fuser -k 8000/tcp 2>/dev/null
!pkill -9 lt 2>/dev/null
!pkill -9 ollama 2>/dev/null
!pkill -9 llama 2>/dev/null
time.sleep(2)

# =========================================================
# 1. PYTHON PACKAGES & BASE SYSTEM SETUP
# =========================================================
print("📥 Installing Python Neural Network libraries...")
!pip install -q \
faster-whisper \
edge-tts \
langgraph \
langchain-ollama \
tavily-python \
soundfile \
uvicorn \
fastapi \
librosa \
ctranslate2

# =========================================================
# 2. ROBUST PYTHON BINARY DOWNLOADER
# =========================================================
target_path = "/usr/local/bin/ollama"
binary_url = "https://github.com/ollama/ollama/releases/download/v0.1.48/ollama-linux-amd64"

print("📥 Downloading Production-Grade Ollama Binary via Python Stream...")
try:
    with urllib.request.urlopen(binary_url) as response, open(target_path, 'wb') as out_file:
        shutil_len = out_file.write(response.read())
    os.chmod(target_path, 0o755)
    print(f"✅ Binary written successfully! Size: {shutil_len / (1024*1024):.2f} MB")
except Exception as download_err:
    print(f"⚠️ Python download failed, falling back to official install script: {download_err}")
    !curl -fsSL https://ollama.com/install.sh | sh

# =========================================================
# 3. ENV CONFIGURATION - SINGLE GPU DEVICE
# =========================================================
print("⚙️ Configuring environment for single T4 GPU (Device 0)...")
os.environ["OLLAMA_HOST"] = "127.0.0.1:11434"
os.environ["OLLAMA_ORIGINS"] = "*"
os.environ["CUDA_VISIBLE_DEVICES"] = "0"           # SINGLE GPU LOCK
os.environ["OLLAMA_NUM_PARALLEL"] = "1"
os.environ["OLLAMA_MAX_LOADED_MODELS"] = "1"
os.environ["OLLAMA_DEBUG"] = "0"

# =========================================================
# 4. INSTANTIATE COGNITIVE SERVER
# =========================================================
print("🚀 Launching Ollama Host Engine on T4 GPU (Device 0)...")
try:
    ollama_env = os.environ.copy()
    ollama_process = subprocess.Popen(
        [target_path, "serve"],
        env=ollama_env,
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL
    )
    time.sleep(8) 
    print("✅ Ollama daemon started successfully")
except Exception as e:
    print(f"❌ Core Exception during instantiation: {e}")

# =========================================================
# 5. VERIFY BINARY & PULL MODEL
# =========================================================
print("🔍 Verifying Ollama binary...")
!/usr/local/bin/ollama --version

target_model = "mistral"

print(f"\n🧠 Pulling agentic model: '{target_model}'...")!/usr/local/bin/ollama list

!/usr/local/bin/ollama pull {target_model}print("\n✅ OLLAMA STACK AND DEPENDENCIES UNIFIED AND STABLE")


🧹 Flushing stale network ports and ghost daemons...
📥 Installing Python Neural Network libraries...
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 17.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 39.0/39.0 MB 53.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 36.3/36.3 MB 52.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 548.1/548.1 kB 25.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 84.1 MB/s eta 0:00:00
📥 Downloading Production-Grade Ollama Binary via Python Stream...
✅ Binary written successfully! Size: 386.16 MB
⚙️ Configuring environment for single T4 GPU (Device 0)...
🚀 Launching Ollama Host Engine on T4 GPU (Device 0)...
✅ Ollama daemon started successfully
🔍 Verifying Ollama binary...
ollama version is 0.1.48

🧠 Pulling agentic model: 'mistral'...
pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠼ pulling manifest ⠼ pulling manifest ⠦ pulling manifest ⠦ pulling manifes

**database.py**

In [ ]:
%%writefile database.py
# This module creates and manages a small SQLite task database.
# It provides helper functions for initializing the database, adding tasks, listing tasks, and clearing tasks.
import sqlite3
import os

DB_PATH = "/kaggle/working/jarvis_memory.db"

def init_db():
    conn = sqlite3.connect(DB_PATH)
    cursor = conn.cursor()
    cursor.execute('''CREATE TABLE IF NOT EXISTS tasks 
                      (id INTEGER PRIMARY KEY AUTOINCREMENT, task TEXT, status TEXT)''')
    conn.commit()
    conn.close()

def get_db_connection():
    return sqlite3.connect(DB_PATH, timeout=20)

def add_task(task_text):
    conn = get_db_connection()
    cursor = conn.cursor()
    cursor.execute("INSERT INTO tasks (task, status) VALUES (?, ?)", (task_text, "pending"))
    conn.commit()
    conn.close()
    return f"Added '{task_text}' to your list, sir."

def list_tasks():
    conn = get_db_connection()
    cursor = conn.cursor()
    cursor.execute("SELECT task FROM tasks")
    tasks = cursor.fetchall()
    conn.close()
    if not tasks: return "Your task list is empty."
    return "Your current tasks are: " + ", ".join([t[0] for t in tasks])

def clear_tasks():
    conn = get_db_connection()
    cursor = conn.cursor()
    cursor.execute("DELETE FROM tasks")
    conn.commit()

    conn.close()    return "I've cleared your task list, sir."

Writing database.py


**tools.py**

In [ ]:
%%writefile tools.py
# This module provides a simple wrapper around the Tavily search API.
# It retrieves secrets from the Kaggle environment and exposes a reusable web_search tool.
import os
from tavily import TavilyClient

# In Kaggle, use 'Secrets' to store your TAVILY_API_KEY
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
TAVILY_KEY = user_secrets.get_secret("TAVILY_API_KEY")

tavily = TavilyClient(api_key=TAVILY_KEY)

def web_search(query: str):

    response = tavily.search(query=query, search_depth="basic")    return "\n".join([f"- {res['content']}" for res in response['results']])

Writing tools.py


**main.py**

In [ ]:
%%writefile main.py
# This is the main FastAPI application for the J.A.R.V.I.S. voice agent.
# It loads speech-to-text, connects to the Ollama LLM via ChatOllama, and exposes a voice processing endpoint.
import os, uuid, torch, asyncio, edge_tts, shutil
from typing import TypedDict, Dict, Any, List
from contextlib import asynccontextmanager

from fastapi import FastAPI, UploadFile, File
from fastapi.middleware.cors import CORSMiddleware
from fastapi.staticfiles import StaticFiles
from fastapi.responses import FileResponse, JSONResponse

from faster_whisper import WhisperModel
from langgraph.graph import StateGraph, END
from langgraph.checkpoint.memory import MemorySaver
from langchain_ollama import ChatOllama
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage
from langchain_core.tools import tool

from database import add_task, list_tasks, clear_tasks, init_db
from tools import web_search

SHM_DIR = "/dev/shm/jarvis_matrix"
OUTPUTS_DIR = os.path.join(SHM_DIR, "outputs")
os.makedirs(OUTPUTS_DIR, exist_ok=True)

class AgentState(TypedDict):
    audio_path: str
    user_speech: str
    ai_response: str
    ai_audio_url: str
    chat_history: List[Any]
    metrics: Dict[str, float]

# =========================================================
# AGENT TOOL MATRIX DEFINITIONS
# =========================================================

@tool
def search_web_tool(query: str) -> str:
    """Search the web for real-time data: current events, news, weather, prices, sports scores."""
    try:
        result = web_search(query)
        return result if result else "No results found."
    except Exception as e:
        return f"Search failed: {e}"

@tool
def add_task_tool(task: str) -> str:
    """Save a task or reminder to the persistent task list."""
    return add_task(task)

@tool
def list_tasks_tool() -> str:
    """Retrieve all saved tasks from the persistent task list."""
    return list_tasks()

@tool
def clear_tasks_tool() -> str:
    """Delete all tasks from the persistent task list."""
    return clear_tasks()

stt_model = None
llm_with_tools = None

def get_llm():
    global llm_with_tools
    if llm_with_tools is None:
        print("🧠 [LAZY INFERENCE] Establishing isolated model links on port 11434...")
        try:
            llm = ChatOllama(
                model="mistral",
                temperature=0.0,
                base_url="http://127.0.0.1:11434",
                num_ctx=1024,
                num_predict=128
            )
            llm_with_tools = llm.bind_tools([
                search_web_tool,
                add_task_tool,
                list_tasks_tool,
                clear_tasks_tool,
            ])
            print("✅ Agent Engine loaded with Mistral execution core + 4 tools")
        except Exception as e:
            print(f"❌ Primary initialization failure: {e}")
    return llm_with_tools

@asynccontextmanager
async def lifespan(app: FastAPI):
    global stt_model
    print("🎯 [SYSTEM INITIALIZATION] J.A.R.V.I.S. Core Boot Sequence...")
    if torch.cuda.is_available():
        torch.backends.cuda.matmul.allow_tf32 = True
        torch.backends.cudnn.benchmark = True
        print("✅ CUDA optimization active")

    print("📦 [STT LOADING] Loading Fast-Whisper onto Nvidia T4 Device [0]...")
    try:
        stt_model = WhisperModel("base.en", device="cuda", device_index=0,
                                  compute_type="float16", cpu_threads=2)
        print("✅ Whisper loaded on GPU 0")
    except Exception as e:
        print(f"⚠️ GPU failed ({e}), using CPU fallback configuration...")
        stt_model = WhisperModel("base.en", device="cpu", compute_type="int8", cpu_threads=4)

    try:
        init_db()
        print("✅ Database initialized")
    except Exception as e:
        print(f"⚠️ Database init warning: {e}")

    print("🚀 [SYSTEM READY] J.A.R.V.I.S. Core systems fully operational\n")
    yield
    if stt_model:
        del stt_model

app = FastAPI(lifespan=lifespan)
app.add_middleware(CORSMiddleware, allow_origins=["*"], allow_methods=["*"], allow_headers=["*"])
app.mount("/audio", StaticFiles(directory=OUTPUTS_DIR), name="audio")

def _extract_task(user_input: str, lowered: str) -> str:
    triggers = ["remind me to", "remind me", "remember to", "remember",
                "add task", "note that", "note", "save that", "add buy", "add by", "add"]
    for phrase in triggers:
        if phrase in lowered:
            idx = lowered.find(phrase) + len(phrase)
            extracted = user_input[idx:].strip(" ,.")
            # Strip out trailing structural words that whisper captures
            for trailing in ["into my list", "in my list", "to my list", "my list"]:
                if extracted.lower().endswith(trailing):
                    extracted = extracted[:-len(trailing)].strip(" ,.")
            return extracted if extracted else user_input
    return user_input

# =========================================================
# SYSTEM PROCESSING NODES
# =========================================================

async def stt_node(state: AgentState) -> Dict[str, Any]:
    start = asyncio.get_event_loop().time()
    try:
        segments, _ = stt_model.transcribe(state["audio_path"], beam_size=1, vad_filter=True)
        text = " ".join([s.text for s in segments]).strip()
        latency = (asyncio.get_event_loop().time() - start) * 1000
        return {"user_speech": text if text else "...", "metrics": {"stt_latency_ms": latency}}
    except Exception as e:
        return {"user_speech": "...", "metrics": {"stt_latency_ms": 0.0}}

async def thinking_node(state: AgentState) -> Dict[str, Any]:
    start = asyncio.get_event_loop().time()
    user_input = state["user_speech"].strip()
    lowered = user_input.lower()
    metrics = state.get("metrics", {})

    # ── 1. Greetings Route (Instant bypass) ─────────────────────────────────
    if lowered in {"hello", "hi", "hey", "hello hello", "jarvis", "online"}:
        metrics["llm_latency_ms"] = 0
        return {"ai_response": "At your service, sir. Systems are fully functional.", "metrics": metrics}

    if user_input == "...":
        metrics["llm_latency_ms"] = 0
        return {"ai_response": "Telemetry warning: No clean audio spectrum detected.", "metrics": metrics}

    # ── 2. Forced Web Search Interceptor (Prioritized globally to prevent structural hijacking) ──
    realtime_keys = [
        "weather", "temperature", "forecast", "google", "internet",
        "news", "latest", "current events", "breaking", "search",
        "score", "match result", "who won", "president",
        "stock", "price of", "bitcoin", "crypto", "look up", "tavily"
    ]
    force_search = any(kw in lowered for kw in realtime_keys)

    if force_search:
        clean_query = user_input
        for filler in ["search for", "look up", "tell me", "what is the", "what's the", 
                       "can you search", "jarvis", "please", "search", "web", "from the internet"]:
            clean_query = clean_query.replace(filler, "").replace(filler.capitalize(), "")
        clean_query = clean_query.strip(" ,.?!")
        
        print(f"📡 [FORCED INTERCEPT] Intercepting query for Tavily: '{clean_query}'")
        try:
            context = await asyncio.to_thread(web_search, clean_query if clean_query else user_input)
            brain = get_llm()
            summary = await brain.ainvoke([
                SystemMessage(content="You are J.A.R.V.I.S. Synthesize the provided search context into a highly professional response of 2-3 sentences max. Answer directly."),
                HumanMessage(content=f"Context:\n{context}\n\nUser Question: {user_input}")
            ])
            res = summary.content
        except Exception as err:
            res = "I encountered an error connecting to search arrays, sir."

        metrics["llm_latency_ms"] = (asyncio.get_event_loop().time() - start) * 1000
        history = list(state.get("chat_history", []))
        history.extend([HumanMessage(content=user_input), AIMessage(content=str(res))])
        return {"ai_response": str(res), "chat_history": history, "metrics": metrics}

    # ── 3. Structural Database Interceptors (Strict Phrase Evaluation Execution) ──
    add_keys   = ["remember", "add task", "remind me", "note that", "note this", "save that", "create task", "add buy", "add by"]
    clear_keys = ["clear tasks", "delete tasks", "wipe tasks", "empty tasks", "clear my list", "delete my list", "wipe list", "clear my lace"]
    list_keys  = ["list tasks", "show tasks", "view tasks", "what are my tasks", "show my tasks", "my tasks", "what's on my list", "read out my list", "what is in my list"]

    # Explicit Execution Sequence: Add/Clear actions are given matching precedence over generic list inquiries
    if any(k in lowered for k in add_keys):
        print("➕ [FORCED INTERCEPT] Direct database injection route verified.")
        task = _extract_task(user_input, lowered)
        res = add_task(task)
        metrics["llm_latency_ms"] = (asyncio.get_event_loop().time() - start) * 1000
        history = list(state.get("chat_history", []))
        history.extend([HumanMessage(content=user_input), AIMessage(content=str(res))])
        return {"ai_response": str(res), "chat_history": history, "metrics": metrics}

    if any(k in lowered for k in clear_keys):
        print("🗑️ [FORCED INTERCEPT] Direct database clear route verified.")
        res = clear_tasks()
        metrics["llm_latency_ms"] = (asyncio.get_event_loop().time() - start) * 1000
        history = list(state.get("chat_history", []))
        history.extend([HumanMessage(content=user_input), AIMessage(content=str(res))])
        return {"ai_response": str(res), "chat_history": history, "metrics": metrics}

    if any(k in lowered for k in list_keys):
        print("📋 [FORCED INTERCEPT] Direct database tracking query verified.")
        res = list_tasks()
        metrics["llm_latency_ms"] = (asyncio.get_event_loop().time() - start) * 1000
        history = list(state.get("chat_history", []))
        history.extend([HumanMessage(content=user_input), AIMessage(content=str(res))])
        return {"ai_response": str(res), "chat_history": history, "metrics": metrics}

    # ── 4. Standard Agentic Path (Fallback conversation context parsing) ──
    system_prompt = SystemMessage(content=(
        "You are J.A.R.V.I.S., an advanced autonomous AI assistant agent framework.\n"
        "Keep conversational answers brief (1-2 sentences)."
    ))

    messages = [system_prompt]
    if state.get("chat_history"):
        messages.extend(state["chat_history"][-4:])
    messages.append(HumanMessage(content=user_input))

    brain = get_llm()
    res = None

    for attempt in range(3):
        try:
            print(f"🧠 [LLM Agent Pipeline attempt {attempt+1}/3] '{user_input[:60]}'")
            ai_msg = await brain.ainvoke(messages)
            text_content = ai_msg.content.lower()
            
            # Sub-layer Text Fallback Routing
            if any(phrase in lowered for phrase in ["in my list", "on my list", "read my list", "what is in my"]):
                print("🔧 [TEXT FALLBACK PARSER] Fallback match: routing to list_tasks_tool")
                res = list_tasks()
            elif any(phrase in lowered for phrase in ["clear", "delete", "wipe"]):
                print("🔧 [TEXT FALLBACK PARSER] Fallback match: routing to clear_tasks_tool")
                res = clear_tasks()
            elif any(phrase in lowered for phrase in ["add", "remember", "remind"]):
                print("🔧 [TEXT FALLBACK PARSER] Fallback match: routing to add_task_tool")
                res = add_task(_extract_task(user_input, lowered))
            else:
                res = ai_msg.content

            print("✅ Agent reasoning loop complete.")
            break
        except Exception as e:
            print(f"⚠️ Reasoning loop exception (Attempt {attempt+1}): {str(e)[:80]}")
            if attempt < 2:
                await asyncio.sleep(2)
            else:
                res = "My internal agent routing loop timed out, sir. Please retry."

    metrics["llm_latency_ms"] = (asyncio.get_event_loop().time() - start) * 1000
    history = list(state.get("chat_history", []))
    history.extend([HumanMessage(content=user_input), AIMessage(content=str(res))])
    return {"ai_response": str(res), "chat_history": history, "metrics": metrics}

async def tts_node(state: AgentState) -> Dict[str, Any]:
    start = asyncio.get_event_loop().time()
    metrics = state.get("metrics", {})
    if not state["ai_response"]:
        return {"ai_audio_url": "", "metrics": metrics}
    try:
        fname = f"{uuid.uuid4()}.mp3"
        path = os.path.join(OUTPUTS_DIR, fname)
        await edge_tts.Communicate(state["ai_response"], "en-GB-RyanNeural").save(path)
        metrics["tts_latency_ms"] = (asyncio.get_event_loop().time() - start) * 1000
        return {"ai_audio_url": fname, "metrics": metrics}
    except Exception as e:
        return {"ai_audio_url": "", "metrics": metrics}

# =========================================================
# GRAPH INTEGRATION ENGINE
# =========================================================
memory_provider = MemorySaver()
workflow = StateGraph(AgentState)
workflow.add_node("stt", stt_node)
workflow.add_node("brain", thinking_node)
workflow.add_node("tts", tts_node)
workflow.set_entry_point("stt")
workflow.add_edge("stt", "brain")
workflow.add_edge("brain", "tts")
workflow.add_edge("tts", END)
graph = workflow.compile(checkpointer=memory_provider)

@app.get("/")
async def serve_dashboard():
    return FileResponse("/kaggle/working/index.html")

@app.post("/process-voice")
async def process_voice(file: UploadFile = File(...)):
    temp_path = os.path.join(SHM_DIR, f"{uuid.uuid4()}.wav")
    with open(temp_path, "wb") as f:
        shutil.copyfileobj(file.file, f)
    execution_config = {"configurable": {"thread_id": "matrix_session_prod"}}
    result = await graph.ainvoke({"audio_path": temp_path}, config=execution_config)
    try:
        os.remove(temp_path)
    except:
        pass
    return JSONResponse(content={
        "text": result.get("ai_response", ""),
        "user_transcription": result.get("user_speech", ""),
        "audio_path": result.get("ai_audio_url", ""),

        "telemetry": result.get("metrics", {})    })

Writing main.py


This HTML file renders the J.A.R.V.I.S. frontend dashboard and voice capture UI.

<!-- This HTML file renders the J.A.R.V.I.S. frontend dashboard and voice capture UI. -->


In [ ]:
%%writefile index.html
<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="UTF-8">
<meta name="viewport" content="width=device-width, initial-scale=1.0">
<title>J.A.R.V.I.S.</title>
<link href="https://fonts.googleapis.com/css2?family=Orbitron:wght@400;600;900&family=JetBrains+Mono:wght@300;400;500&display=swap" rel="stylesheet">
<style>
*, *::before, *::after { box-sizing: border-box; margin: 0; padding: 0; }

:root {
  --bg:      #040810;
  --bg1:     #070d1a;
  --bg2:     #0a1220;
  --border:  rgba(0,210,255,0.12);
  --border2: rgba(0,210,255,0.25);
  --cyan:    #00d2ff;
  --cyan2:   #00a8cc;
  --purple:  #a78bfa;
  --amber:   #fbbf24;
  --red:     #ff4466;
  --green:   #22d3a5;
  --text:    #c8daf0;
  --muted:   #4a6080;
  --mono:    'JetBrains Mono', monospace;
  --display: 'Orbitron', sans-serif;
}

body {
  font-family: var(--mono);
  background: var(--bg);
  color: var(--text);
  min-height: 100vh;
  display: grid;
  grid-template-rows: auto 1fr auto;
  overflow: hidden;
}

/* Animated grid background */
body::before {
  content: '';
  position: fixed;
  inset: 0;
  background-image:
    linear-gradient(rgba(0,210,255,0.03) 1px, transparent 1px),
    linear-gradient(90deg, rgba(0,210,255,0.03) 1px, transparent 1px);
  background-size: 40px 40px;
  pointer-events: none;
  z-index: 0;
}

/* Subtle vignette */
body::after {
  content: '';
  position: fixed;
  inset: 0;
  background: radial-gradient(ellipse at center, transparent 40%, rgba(4,8,16,0.8) 100%);
  pointer-events: none;
  z-index: 0;
}

header, main, footer { position: relative; z-index: 1; }

/* ── HEADER ── */
header {
  display: flex;
  justify-content: space-between;
  align-items: center;
  padding: 14px 28px;
  border-bottom: 1px solid var(--border);
  background: rgba(7,13,26,0.9);
  backdrop-filter: blur(12px);
}

.logo-group h1 {
  font-family: var(--display);
  font-size: 22px;
  font-weight: 900;
  letter-spacing: 0.2em;
  color: var(--cyan);
  text-shadow: 0 0 20px rgba(0,210,255,0.5), 0 0 40px rgba(0,210,255,0.2);
  line-height: 1;
}

.logo-group p {
  font-size: 9px;
  letter-spacing: 0.15em;
  color: var(--muted);
  margin-top: 3px;
}

.header-stats {
  display: flex;
  gap: 28px;
  align-items: flex-end;
}

.stat-item { text-align: right; }
.stat-label { font-size: 9px; letter-spacing: 0.12em; color: var(--muted); display: block; margin-bottom: 2px; }
.stat-value { font-family: var(--display); font-size: 11px; font-weight: 600; letter-spacing: 0.08em; }
.stat-value.cyan   { color: var(--cyan); }
.stat-value.green  { color: var(--green); }

.pulse-dot {
  display: inline-block;
  width: 6px; height: 6px;
  border-radius: 50%;
  background: var(--green);
  box-shadow: 0 0 8px var(--green);
  animation: pulse 2s ease-in-out infinite;
  margin-right: 5px;
  vertical-align: middle;
}

@keyframes pulse { 0%,100%{opacity:1} 50%{opacity:0.3} }

/* ── MAIN GRID ── */
main {
  display: grid;
  grid-template-columns: 300px 1fr;
  gap: 0;
  height: calc(100vh - 96px);
  overflow: hidden;
}

/* ── PANEL BASE ── */
.panel {
  border-right: 1px solid var(--border);
  padding: 20px;
  display: flex;
  flex-direction: column;
  gap: 16px;
  overflow: hidden;
}

.panel-title {
  font-family: var(--display);
  font-size: 9px;
  font-weight: 600;
  letter-spacing: 0.2em;
  color: var(--muted);
  padding-bottom: 10px;
  border-bottom: 1px solid var(--border);
}

/* ── LEFT: INPUT PANEL ── */
#input-panel {
  background: linear-gradient(180deg, rgba(0,210,255,0.02) 0%, transparent 100%);
}

/* Orb */
.orb-wrap {
  display: flex;
  flex-direction: column;
  align-items: center;
  gap: 16px;
  padding: 24px 0 20px;
}

.orb-ring {
  position: relative;
  width: 120px;
  height: 120px;
}

/* outer ring — always visible */
.orb-ring::before {
  content: '';
  position: absolute;
  inset: -6px;
  border-radius: 50%;
  border: 1px solid var(--border2);
}

/* spinning segment — only when recording */
.orb-ring::after {
  content: '';
  position: absolute;
  inset: -6px;
  border-radius: 50%;
  border: 2px solid transparent;
  border-top-color: var(--cyan);
  animation: spin 1.2s linear infinite;
  opacity: 0;
  transition: opacity 0.3s;
}

.orb-ring.recording::after { opacity: 1; }
.orb-ring.processing::after { border-top-color: var(--amber); opacity: 1; animation-duration: 0.6s; }

@keyframes spin { to { transform: rotate(360deg); } }

.orb {
  width: 120px;
  height: 120px;
  border-radius: 50%;
  background: radial-gradient(circle at 35% 35%, rgba(0,210,255,0.15), rgba(0,210,255,0.04) 60%, transparent);
  border: 1px solid var(--border2);
  display: flex;
  align-items: center;
  justify-content: center;
  transition: all 0.3s ease;
  cursor: default;
  position: relative;
  overflow: hidden;
}

.orb::before {
  content: '';
  position: absolute;
  inset: 0;
  border-radius: 50%;
  background: radial-gradient(circle at 35% 35%, rgba(255,255,255,0.06), transparent 60%);
}

.orb-inner {
  width: 32px;
  height: 32px;
  border-radius: 50%;
  background: var(--cyan);
  opacity: 0.6;
  box-shadow: 0 0 20px rgba(0,210,255,0.4);
  transition: all 0.3s ease;
}

.orb.recording {
  border-color: var(--red);
  background: radial-gradient(circle at 35% 35%, rgba(255,68,102,0.12), rgba(255,68,102,0.04) 60%, transparent);
  box-shadow: 0 0 30px rgba(255,68,102,0.2);
}
.orb.recording .orb-inner {
  background: var(--red);
  box-shadow: 0 0 20px rgba(255,68,102,0.5);
  opacity: 1;
  animation: breathe 0.8s ease-in-out infinite;
}
.orb.processing {
  border-color: var(--amber);
}
.orb.processing .orb-inner {
  background: var(--amber);
  box-shadow: 0 0 20px rgba(251,191,36,0.5);
  opacity: 0.8;
}

@keyframes breathe {
  0%,100% { transform: scale(1); }
  50% { transform: scale(1.25); }
}

.orb-status {
  font-size: 10px;
  letter-spacing: 0.18em;
  color: var(--muted);
  text-align: center;
  min-height: 14px;
  transition: color 0.3s;
}

.orb-status.active { color: var(--cyan); }
.orb-status.recording { color: var(--red); }
.orb-status.processing { color: var(--amber); }

/* Waveform bars */
.waveform {
  display: flex;
  align-items: center;
  gap: 3px;
  height: 32px;
  justify-content: center;
}

.waveform-bar {
  width: 3px;
  border-radius: 2px;
  background: var(--cyan2);
  height: 4px;
  transition: height 0.08s ease, background 0.3s;
  min-height: 4px;
}

/* VU meter */
.vu-wrap {
  background: rgba(0,0,0,0.3);
  border: 1px solid var(--border);
  border-radius: 3px;
  padding: 10px 12px;
}

.vu-label-row {
  display: flex;
  justify-content: space-between;
  font-size: 9px;
  letter-spacing: 0.1em;
  color: var(--muted);
  margin-bottom: 6px;
}

.vu-track {
  height: 3px;
  background: rgba(255,255,255,0.05);
  border-radius: 2px;
  overflow: visible;
  position: relative;
  margin-bottom: 6px;
}

.vu-fill {
  height: 100%;
  border-radius: 2px;
  background: linear-gradient(90deg, var(--cyan2), var(--cyan));
  width: 0%;
  transition: width 0.06s linear;
  position: relative;
}

/* threshold marker */
.vu-threshold {
  position: absolute;
  top: -4px;
  bottom: -4px;
  width: 1px;
  background: rgba(251,191,36,0.6);
  /* positioned by JS */
}

.vu-db {
  font-family: var(--display);
  font-size: 11px;
  color: var(--cyan);
  text-align: right;
  letter-spacing: 0.05em;
}

/* VAD config */
.vad-config {
  background: rgba(0,0,0,0.25);
  border: 1px solid var(--border);
  border-radius: 3px;
  padding: 10px 12px;
  display: flex;
  flex-direction: column;
  gap: 8px;
}

.config-row {
  display: flex;
  justify-content: space-between;
  align-items: center;
  font-size: 9px;
  letter-spacing: 0.1em;
  color: var(--muted);
}

.config-row input[type=range] {
  width: 80px;
  accent-color: var(--cyan);
  cursor: pointer;
}

.config-val {
  font-family: var(--display);
  font-size: 10px;
  color: var(--cyan);
  min-width: 44px;
  text-align: right;
}

/* ── RIGHT: CHAT PANEL ── */
#chat-panel {
  border-right: none;
  background: linear-gradient(180deg, rgba(167,139,250,0.02) 0%, transparent 100%);
}

/* Telemetry bar */
.telemetry-row {
  display: grid;
  grid-template-columns: repeat(3, 1fr);
  gap: 10px;
  flex-shrink: 0;
}

.tele-card {
  background: rgba(0,0,0,0.3);
  border: 1px solid var(--border);
  border-radius: 3px;
  padding: 8px 10px;
}

.tele-label { font-size: 8px; letter-spacing: 0.12em; color: var(--muted); margin-bottom: 3px; }
.tele-val {
  font-family: var(--display);
  font-size: 15px;
  font-weight: 600;
  letter-spacing: 0.03em;
  line-height: 1;
}
.tele-unit { font-size: 9px; font-weight: 300; opacity: 0.6; }
.tele-val.c { color: var(--cyan); }
.tele-val.p { color: var(--purple); }
.tele-val.a { color: var(--amber); }

/* Chat log */
.chat-log {
  flex: 1;
  overflow-y: auto;
  display: flex;
  flex-direction: column;
  gap: 10px;
  padding-right: 4px;
  scrollbar-width: thin;
  scrollbar-color: var(--border2) transparent;
}

.chat-log::-webkit-scrollbar { width: 4px; }
.chat-log::-webkit-scrollbar-track { background: transparent; }
.chat-log::-webkit-scrollbar-thumb { background: var(--border2); border-radius: 2px; }

.msg {
  border-left: 2px solid;
  padding: 10px 12px;
  border-radius: 0 3px 3px 0;
  font-size: 12px;
  line-height: 1.6;
  background: rgba(0,0,0,0.2);
  animation: fadeIn 0.2s ease;
}

@keyframes fadeIn { from { opacity:0; transform: translateY(4px); } to { opacity:1; transform: none; } }

.msg-label {
  font-size: 9px;
  letter-spacing: 0.14em;
  font-weight: 500;
  margin-bottom: 4px;
  display: block;
}

.msg.user   { border-color: var(--cyan2); }
.msg.user   .msg-label { color: var(--cyan); }
.msg.jarvis { border-color: var(--purple); }
.msg.jarvis .msg-label { color: var(--purple); }
.msg.system { border-color: var(--border2); }
.msg.system .msg-label { color: var(--muted); }
.msg.system { color: var(--muted); font-size: 11px; }

/* ── FOOTER ── */
footer {
  padding: 8px 28px;
  border-top: 1px solid var(--border);
  background: rgba(7,13,26,0.9);
  display: flex;
  justify-content: space-between;
  font-size: 9px;
  letter-spacing: 0.12em;
  color: var(--muted);
}
</style>
</head>
<body>

<header>
  <div class="logo-group">
    <h1>J.A.R.V.I.S.</h1>
    <p>JUST A RATHER VERY INTELLIGENT SYSTEM // AGENT MATRIX v3.1</p>
  </div>
  <div class="header-stats">
    <div class="stat-item">
      <span class="stat-label">CORE COGNITION</span>
      <span class="stat-value cyan">NEURAL-CHAT // LOCAL</span>
    </div>
    <div class="stat-item">
      <span class="stat-label">SYSTEM STATUS</span>
      <span class="stat-value green"><span class="pulse-dot"></span>ONLINE</span>
    </div>
  </div>
</header>

<main>

  <!-- LEFT: INPUT -->
  <div class="panel" id="input-panel">
    <div class="panel-title">// VOICE CAPTURE ENGINE</div>

    <div class="orb-wrap">
      <div class="orb-ring" id="orb-ring">
        <div class="orb" id="orb">
          <div class="orb-inner" id="orb-inner"></div>
        </div>
      </div>

      <!-- Waveform visualiser -->
      <div class="waveform" id="waveform">
        <!-- bars injected by JS -->
      </div>

      <div class="orb-status" id="orb-status">ARMING VAD ENGINE...</div>
    </div>

    <!-- VU meter -->
    <div class="vu-wrap">
      <div class="vu-label-row">
        <span>AUDIO LEVEL</span>
        <span id="vu-db">— dB</span>
      </div>
      <div class="vu-track">
        <div class="vu-fill" id="vu-fill"></div>
        <div class="vu-threshold" id="vu-threshold"></div>
      </div>
    </div>

    <!-- VAD controls -->
    <div class="vad-config">
      <div class="config-row">
        <span>TRIGGER THRESHOLD</span>
        <div style="display:flex;align-items:center;gap:6px">
          <input type="range" id="slider-thresh" min="-70" max="-20" value="-42" step="1">
          <span class="config-val" id="val-thresh">-42 dB</span>
        </div>
      </div>
      <div class="config-row">
        <span>SILENCE TIMEOUT</span>
        <div style="display:flex;align-items:center;gap:6px">
          <input type="range" id="slider-silence" min="400" max="2500" value="1100" step="100">
          <span class="config-val" id="val-silence">1.1 s</span>
        </div>
      </div>
      <div class="config-row">
        <span>NOISE GATE FRAMES</span>
        <div style="display:flex;align-items:center;gap:6px">
          <input type="range" id="slider-gate" min="2" max="12" value="4" step="1">
          <span class="config-val" id="val-gate">4</span>
        </div>
      </div>
    </div>

    <!-- info -->
    <div style="font-size:9px;letter-spacing:0.1em;color:var(--muted);line-height:1.8;margin-top:auto">
      <div>[ VAD ] AUTO-TRIGGER ON VOICE ACTIVITY</div>
      <div>[ LOCK ] MUTED DURING PLAYBACK</div>
    </div>
  </div>

  <!-- RIGHT: CHAT -->
  <div class="panel" id="chat-panel">
    <div class="panel-title">// QUANTUM RESPONSE MONITOR</div>

    <div class="telemetry-row">
      <div class="tele-card">
        <div class="tele-label">ASR TRANSCRIPTION</div>
        <div class="tele-val c" id="metric-stt">—<span class="tele-unit"> ms</span></div>
      </div>
      <div class="tele-card">
        <div class="tele-label">INFERENCE PIPELINE</div>
        <div class="tele-val p" id="metric-llm">—<span class="tele-unit"> ms</span></div>
      </div>
      <div class="tele-card">
        <div class="tele-label">NEURAL SPEECH</div>
        <div class="tele-val a" id="metric-tts">—<span class="tele-unit"> ms</span></div>
      </div>
    </div>

    <div class="chat-log" id="chat-log">
      <div class="msg system">
        <span class="msg-label">[SYSTEM]</span>
        VAD active. Speak naturally — recording triggers automatically on voice detection.
      </div>
    </div>
  </div>

</main>

<audio id="vocal-player" class="hidden" style="display:none"></audio>

<footer>
  <span>NATIONAL TEXTILE UNIVERSITY // BSAI SEMESTER 6</span>
  <span id="clock">--:--:--</span>
</footer>

<script>
const BACKEND_URL = window.location.origin;

// ── State ──────────────────────────────────────────────────────────────────
let mediaRecorder, audioChunks = [], isRecording = false, isProcessingAPI = false;
let audioContext, analyser, scriptProc;

// VAD params (user-adjustable)
let VAD_THRESHOLD   = -42;   // dB
let SILENCE_MS      = 1100;  // ms
let NOISE_GATE_FRAMES = 4;   // consecutive above-threshold frames required to start

let silenceStart  = null;
let aboveCount    = 0;       // noise gate counter
let smoothedDb    = -100;    // exponential smoothed dB

// Waveform bars
const BARS = 20;
const waveformEl = document.getElementById('waveform');
const bars = [];
for (let i = 0; i < BARS; i++) {
  const b = document.createElement('div');
  b.className = 'waveform-bar';
  waveformEl.appendChild(b);
  bars.push(b);
}

// UI refs
const orb       = document.getElementById('orb');
const orbRing   = document.getElementById('orb-ring');
const orbStatus = document.getElementById('orb-status');
const vuFill    = document.getElementById('vu-fill');
const vuDb      = document.getElementById('vu-db');
const vuThresh  = document.getElementById('vu-threshold');
const chatLog   = document.getElementById('chat-log');
const vocalPlayer = document.getElementById('vocal-player');

// Position threshold marker
function updateThresholdMarker() {
  const pct = Math.max(0, Math.min(100, (VAD_THRESHOLD + 90) * (100 / 70)));
  vuThresh.style.left = pct + '%';
}
updateThresholdMarker();

// Sliders
document.getElementById('slider-thresh').addEventListener('input', e => {
  VAD_THRESHOLD = parseInt(e.target.value);
  document.getElementById('val-thresh').textContent = VAD_THRESHOLD + ' dB';
  updateThresholdMarker();
});
document.getElementById('slider-silence').addEventListener('input', e => {
  SILENCE_MS = parseInt(e.target.value);
  document.getElementById('val-silence').textContent = (SILENCE_MS / 1000).toFixed(1) + ' s';
});
document.getElementById('slider-gate').addEventListener('input', e => {
  NOISE_GATE_FRAMES = parseInt(e.target.value);
  document.getElementById('val-gate').textContent = NOISE_GATE_FRAMES;
});

// Clock
function updateClock() {
  document.getElementById('clock').textContent = new Date().toTimeString().slice(0, 8);
}
setInterval(updateClock, 1000);
updateClock();

// ── VAD Setup ──────────────────────────────────────────────────────────────
window.addEventListener('DOMContentLoaded', async () => {
  try {
    const stream = await navigator.mediaDevices.getUserMedia({ audio: true, video: false });

    audioContext = new (window.AudioContext || window.webkitAudioContext)({ sampleRate: 16000 });
    analyser     = audioContext.createAnalyser();
    analyser.fftSize              = 1024;
    analyser.smoothingTimeConstant = 0.0; // we smooth manually for finer control

    const src = audioContext.createMediaStreamSource(stream);
    scriptProc  = audioContext.createScriptProcessor(1024, 1, 1);

    src.connect(analyser);
    analyser.connect(scriptProc);
    scriptProc.connect(audioContext.destination);

    mediaRecorder = new MediaRecorder(stream, { mimeType: 'audio/webm' });
    mediaRecorder.ondataavailable = e => audioChunks.push(e.data);
    mediaRecorder.onstop = async () => {
      const blob = new Blob(audioChunks, { type: 'audio/webm' });
      await transmit(blob);
    };

    // Frequency data buffer for waveform
    const freqBuf = new Uint8Array(analyser.frequencyBinCount);
    const timeBuf = new Float32Array(analyser.fftSize);

    scriptProc.onaudioprocess = () => {
      if (isProcessingAPI) {
        // clear UI while locked
        bars.forEach(b => b.style.height = '4px');
        vuFill.style.width = '0%';
        vuDb.textContent = 'LOCKED';
        return;
      }

      // ── Compute RMS dB from time-domain ───────────────────────────
      analyser.getFloatTimeDomainData(timeBuf);
      let sum = 0;
      for (let i = 0; i < timeBuf.length; i++) sum += timeBuf[i] * timeBuf[i];
      const rms  = Math.sqrt(sum / timeBuf.length);
      const rawDb = rms > 0 ? 20 * Math.log10(rms) : -100;

      // Exponential smoothing (fast attack, slow decay)
      const alpha = rawDb > smoothedDb ? 0.6 : 0.15;
      smoothedDb  = alpha * rawDb + (1 - alpha) * smoothedDb;
      const db    = Math.max(-100, Math.min(0, smoothedDb));

      // ── VU meter ──────────────────────────────────────────────────
      const pct = Math.max(0, Math.min(100, (db + 90) * (100 / 70)));
      vuFill.style.width  = pct + '%';
      vuDb.textContent    = db.toFixed(1) + ' dB';

      // ── Waveform bars from freq domain ────────────────────────────
      analyser.getByteFrequencyData(freqBuf);
      const step = Math.floor(freqBuf.length / BARS);
      for (let i = 0; i < BARS; i++) {
        const val = freqBuf[i * step] / 255;
        const h   = Math.max(4, val * 28);
        bars[i].style.height = h + 'px';
        bars[i].style.background = isRecording
          ? `rgba(255,68,102,${0.4 + val * 0.6})`
          : `rgba(0,168,204,${0.3 + val * 0.5})`;
      }

      // ── VAD State Machine ─────────────────────────────────────────
      if (db > VAD_THRESHOLD) {
        silenceStart = null;
        aboveCount   = Math.min(aboveCount + 1, NOISE_GATE_FRAMES + 1);

        if (!isRecording && aboveCount >= NOISE_GATE_FRAMES) {
          startRecording();
        }
      } else {
        aboveCount = 0; // reset gate on any quiet frame

        if (isRecording) {
          if (!silenceStart) silenceStart = Date.now();
          if (Date.now() - silenceStart > SILENCE_MS) {
            stopRecording();
          }
        }
      }
    };

    setStatus('idle', 'DEVICE IDLE // VAD ACTIVE');
  } catch (err) {
    console.error(err);
    setStatus('idle', 'MIC ACCESS DENIED');
  }
});

// ── Recording ──────────────────────────────────────────────────────────────
function startRecording() {
  if (isRecording) return;
  audioChunks = [];
  mediaRecorder.start();
  isRecording = true;
  setStatus('recording', 'CAPTURING AUDIO STREAM');
}

function stopRecording() {
  if (!isRecording) return;
  mediaRecorder.stop();
  isRecording = false;
  silenceStart = null;
  aboveCount   = 0;
  isProcessingAPI = true;
  setStatus('processing', 'TRANSMITTING TO BACKEND...');
}

// ── API call ───────────────────────────────────────────────────────────────
async function transmit(blob) {
  const fd = new FormData();
  fd.append('file', blob, 'audio.webm');
  try {
    const res  = await fetch(`${BACKEND_URL}/process-voice`, { method:'POST', body: fd });
    const data = await res.json();

    appendMsg('USER',   data.user_transcription || '(no transcription)', 'user');
    appendMsg('JARVIS', data.text               || '(no response)',       'jarvis');

    if (data.telemetry) {
      const t = data.telemetry;
      setMetric('metric-stt', t.stt_latency_ms);
      setMetric('metric-llm', t.llm_latency_ms);
      setMetric('metric-tts', t.tts_latency_ms);
    }

    if (data.audio_path) {
      vocalPlayer.src = `${BACKEND_URL}/audio/${data.audio_path}`;
      setStatus('processing', 'PLAYING RESPONSE...');
      vocalPlayer.onended = () => unlock();
      vocalPlayer.play().catch(() => unlock());
    } else {
      unlock();
    }
  } catch (err) {
    console.error(err);
    appendMsg('SYSTEM', 'Backend communication error.', 'system');
    unlock();
  }
}

function unlock() {
  isProcessingAPI = false;
  setStatus('idle', 'DEVICE IDLE // VAD ACTIVE');
}

// ── UI helpers ─────────────────────────────────────────────────────────────
function setStatus(state, text) {
  orbStatus.textContent = text;
  orbStatus.className   = 'orb-status ' + (state === 'idle' ? 'active' : state);
  orb.className         = 'orb ' + (state !== 'idle' ? state : '');
  orbRing.className     = 'orb-ring ' + (state !== 'idle' ? state : '');
}

function appendMsg(label, text, cls) {
  const el = document.createElement('div');
  el.className = 'msg ' + cls;
  el.innerHTML = `<span class="msg-label">[${label}]</span>${escHtml(text)}`;
  chatLog.appendChild(el);
  chatLog.scrollTop = chatLog.scrollHeight;
}

function escHtml(s) {
  return String(s).replace(/&/g,'&amp;').replace(/</g,'&lt;').replace(/>/g,'&gt;');
}

function setMetric(id, val) {
  const el = document.getElementById(id);
  if (el && val != null) {
    const unit = el.querySelector('.tele-unit');
    el.innerHTML = '';
    el.appendChild(document.createTextNode(val.toFixed(0)));
    const u = document.createElement('span');
    u.className = 'tele-unit';
    u.textContent = ' ms';
    el.appendChild(u);
  }
}
</script>
</body>
</html>

Writing index.html


In [ ]:
# This helper cell configures and launches the Ollama daemon and FastAPI server, then creates a public tunnel.
import os, time, subprocess

print("🧹 [SYSTEM RECOVERY] Deep flushing ports, processes, and runtime state cache frames...")
!fuser -k 8000/tcp 2>/dev/null
!fuser -k 7860/tcp 2>/dev/null
!fuser -k 11434/tcp 2>/dev/null
!pkill -9 lt 2>/dev/null
!pkill -9 ollama 2>/dev/null
!pkill -9 llama 2>/dev/null
time.sleep(3)

print("📥 [MODEL PRELOAD] Checking optimized model for T4...")
preload_env = os.environ.copy()
preload_env["CUDA_VISIBLE_DEVICES"] = "0"
preload_env["OLLAMA_HOST"] = "127.0.0.1:11434"
preload_env["OLLAMA_NUM_PARALLEL"] = "1"

# Start Ollama temporarily to pull the model
temp_ollama = subprocess.Popen(
    ["/usr/local/bin/ollama", "serve"],
    env=preload_env,
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL
)
time.sleep(5)

model_name = "mistral"
print(f"🔄 Verifying local registry cache parameters for: {model_name}...")

pull_process = subprocess.run(
    ["/usr/local/bin/ollama", "pull", model_name],
    env=preload_env,
    capture_output=True,
    text=True
)

if pull_process.returncode == 0:
    print(f"✅ Model '{model_name}' verified and prepared successfully!")
else:
    print(f"⚠️ Model pull warning: {pull_process.stderr[:100]}")

temp_ollama.terminate()
try:
    temp_ollama.wait(timeout=5)
except:
    temp_ollama.kill()
time.sleep(2)

print("\n🚀 [LAUNCH] Initializing clean Ollama daemon with aggressive memory optimization...")
ollama_env = os.environ.copy()
ollama_env["CUDA_VISIBLE_DEVICES"] = "0"
ollama_env["OLLAMA_HOST"] = "127.0.0.1:11434"
ollama_env["OLLAMA_ORIGINS"] = "*"
ollama_env["OLLAMA_NUM_PARALLEL"] = "1"
ollama_env["OLLAMA_MAX_LOADED_MODELS"] = "1"
ollama_env["OLLAMA_KEEP_ALIVE"] = "5m"

subprocess.Popen(
    ["/usr/local/bin/ollama", "serve"],
    env=ollama_env,
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL
)
print("⏳ Waiting for Ollama daemon to stabilize...")
time.sleep(10)

print("🚀 [LAUNCH] Initializing Unified J.A.R.V.I.S. FastAPI Core on Port 8000...")
def start_server():
    import uvicorn
    uvicorn.run("main:app", host="0.0.0.0", port=8000, log_level="error")

import threading
server_thread = threading.Thread(target=start_server, daemon=True)
server_thread.start()
print("⏳ Waiting for FastAPI server to initialize...")
time.sleep(8)

print("🌐 [TUNNEL] Provisioning localtunnel proxy layer...")
!npm install -g localtunnel -q

print("\n================== PRESENTATION VECTOR ACTIVE ==================")
process = subprocess.Popen(["lt", "--port", "8000"], stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)

timeout_counter = 0
max_timeout = 30

while timeout_counter < max_timeout:
    try:
        line = process.stdout.readline()
        if "url is" in line:
            clean_url = line.strip().replace("your url is: ", "").strip()
            print(f"\n🎯 CORE INTERFACE LINK: {clean_url}")
            print("⚡ System hardware runtime is fully isolated and operational.")
            print(f"✅ Native Model '{model_name}' loaded with optimized memory footprint.")
            break
        if not line:
            timeout_counter += 1
            time.sleep(0.5)
    except Exception as e:
        print(f"⚠️ Tunnel read error: {e}")
        timeout_counter += 1
        time.sleep(0.5)

if timeout_counter >= max_timeout:
    print("❌ System network tunnel configuration failure or timeout.")

🧹 [SYSTEM RECOVERY] Deep flushing ports, processes, and runtime state cache frames...
    63📥 [MODEL PRELOAD] Checking optimized model for T4...
🔄 Verifying local registry cache parameters for: mistral...
✅ Model 'mistral' verified and prepared successfully!

🚀 [LAUNCH] Initializing clean Ollama daemon with aggressive memory optimization...
⏳ Waiting for Ollama daemon to stabilize...
🚀 [LAUNCH] Initializing Unified J.A.R.V.I.S. FastAPI Core on Port 8000...
⏳ Waiting for FastAPI server to initialize...
🌐 [TUNNEL] Provisioning localtunnel proxy layer...
⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼
added 22 packages in 3s
⠼
⠼3 packages are looking for funding
⠼  run `npm fund` for details
⠼npm notice
npm notice New major version of npm available! 10.8.2 -> 11.14.1
npm notice Changelog: https://github.com/npm/cli/releases/tag/v11.14.1
npm notice To update run: npm install -g npm@11.14.1
npm notice
⠼
================== PRESENTATION VECTOR ACTIVE ==================

🎯 CORE INTERFACE LINK: https://eleven-apples-melt.loca.